## Mini tutorial: Deploying code and packages to Dask workers

Prerequisites
- A running Dask Distributed cluster or LocalCluster with a connected Client.
- Basic familiarity with client.run and client.upload_file.



In [ ]:
from distributed import Client


# 1) Attach to a Dask client (local cluster)
client = Client()  # or Client('address:port') for an existing cluster
print("Dask client:", client)

### Check environment consistency across workers
Goal: see Python version, interpreter, and basic environment info to decide if you should install packages or rebuild the environment.


In [ ]:
def env_info():
    import sys, platform
    return {
        "py_version": sys.version,
        "executable": sys.executable,
        "platform": platform.platform(),
    }

# Run on all workers and collect results
envs = client.run(env_info)
print(envs)


### pip Install a package on all workers (if needed)
- Use a Python-friendly invocation to ensure you’re using the same Python interpreter across workers.
- Tip: pin versions to avoid drift.


In [ ]:
def install_pkg(pkg_name):
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg_name])
    return True

# Example: install a package on every worker
install_pkg("gaiadr3-zeropoint") 

# Optional: install with a requirements file
# def install_from_requirements(req_file):
#     import sys, subprocess
#     subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", req_file])
# client.run(install_from_requirements, "requirements.txt")


### Verify the package import on workers
- After installation, test that you can import and use the package on each worker.


In [ ]:

def check_import_zero_point():
    try:
        import zero_point  # or the appropriate import name for your package
        return True
    except Exception:
        return False

results = client.run(check_import_zero_point)
print(results)


For better resilience, you can wrap in a more detailed try/except and return error messages.



### Upload your own code to workers (distribute custom scripts)

Use upload_file to copy a local Python file to each worker. It’s added to the workers’ Python path, so you can import it by name.


In [ ]:

# Upload a local script to workers
client.upload_file("src.py")

# Then on workers you can import and run something from that module
def import_src():
    import src  # if you uploaded src.py, this should work
    return True

src_ok = client.run(import_src)
print("src.py available on workers:", src_ok)



Tips and best practices
- Prefer explicit per-interpreter pip: use python -m pip to ensure you’re using the same Python environment as the worker.
- Idempotence: If you run install multiple times, ensure your installer is robust (e.g., use --upgrade or check if already installed). The simple install function above will raise if it fails, which surfaces issues early.
- Environment management: For reproducibility, build the environment (conda/venv or Docker) ahead of time and start workers with that image. Installing on-the-fly is convenient but can lead to drift and longer startup times.
- Requirements handling: If you have a requirements.txt, consider uploading it and running pip install -r requirements.txt on all workers, or build a small helper that only installs missing packages.
- Security: Be mindful of executing arbitrary code on workers. Only run code from trusted sources and consider sandboxing in multi-user environments.
- Performance: frequent per-worker installs during a long session can be slow. If possible, bake dependencies into the worker images or use a persistent cluster with pre-installed packages.
- Diagnostics: Use client.get_versions() and check the Dask dashboard (especially the Workers tab) to monitor which workers have which packages loaded, memory usage, etc.

